In [1]:
import pandas as pd
!pip install pandas langchain langchain-community langchain-experimental langchain-mistralai faiss-cpu -q
!pip install requests==2.32.4 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
l

In [2]:
import pandas as pd
from google.colab import files
from getpass import getpass

from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_experimental.agents import create_pandas_dataframe_agent

/tmp/ipykernel_10850/2948168083.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/tmp/ipykernel_10850/2948168083.py:8: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents import create_pandas_dataframe_agent


In [3]:
#Subir y leer el dataset

uploaded = files.upload()

df = pd.read_csv("sales_data_normalizado.csv")

df.head()

##Configurar MIstral


api_key = getpass("Pegá tu API Key de Mistral: ")

llm = ChatMistralAI(
    model="mistral-small-latest",
    mistral_api_key=api_key,
    temperature=0
)

embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=api_key
)

##Convertir el Dataframe en corpus para mistral
documentos = []

for index, row in df.iterrows():
    texto_fila = " | ".join([f"{columna}: {row[columna]}" for columna in df.columns])

    documento = Document(
        page_content=texto_fila,
        metadata={"fila": index}
    )

    documentos.append(documento)

print(f"Documentos creados: {len(documentos)}")


##Crear base de datos vectorial
vectorstore = FAISS.from_documents(
    documentos,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

##Crear agente de pandas
pandas_agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    allow_dangerous_code=True
)
##Crear agente RAG + pandas
def agente_rag_pandas(pregunta):
    documentos_relevantes = retriever.invoke(pregunta)

    contexto = "\n\n".join([
        doc.page_content for doc in documentos_relevantes
    ])

    prompt = f"""
Eres un agente de análisis de datos.

Tienes dos fuentes de información:

1. Contexto recuperado mediante RAG:
{contexto}

2. Un DataFrame de pandas llamado df.

Pregunta del usuario:
{pregunta}

Instrucciones:
- Si la pregunta requiere cálculo exacto, usa pandas.
- Si la pregunta requiere contexto o explicación, usa el contexto recuperado.
- Responde en español, claro y breve.
- Recuerda que el dataset está normalizado, por eso muchos valores están entre 0 y 1.
"""

    respuesta = llm.invoke(prompt)

    return respuesta.content


##Probar agente RAG
respuesta = agente_rag_pandas("¿Qué información relevante hay sobre las ventas?")
print(respuesta)

Saving sales_data_normalizado.csv to sales_data_normalizado.csv
Pegá tu API Key de Mistral: ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

Documentos creados: 2823
### **Análisis de Ventas (Información Relevante)**

#### **1. Contexto Recuperado (RAG)**
- **Clientes principales**:
  - **Iberia Gift Imports, Corp.** (Sevilla, España): Realizó **3 pedidos** (todos en 2003, estado *"Shipped"*), con un tamaño de pedido *"Medium"*.
  - **Euro Shopping Channel** (Madrid, España): Realizó **2 pedidos** (en 2005, estado *"In Process"*), con un tamaño de pedido *"Large"*.

- **Productos más vendidos**:
  - Todos pertenecen a la línea **"Trucks and Buses"** (códigos: `S50_1392`, `S12_1666`, `S18_2319`, `S12_4473`, `S18_1097`).
  - **Precios**: Todos con `PRICEEACH = 1.0` (normalizado).

- **Tendencias temporales**:
  - **2003**: 3 pedidos (todos en el **Q1**, noviembre).
  - **2005**: 2 pedidos (en el **Q2**, mayo).

- **Territorio**: Ambos clientes son de **EMEA** (Europa, Oriente Medio y África).

---

#### **2. Cálculos con el DataFrame (df)**
*(Si el usuario requiere métricas exactas, se pueden calcular con pandas. Ejemplos com